# 00C Demo Setup — FabricOps I/O

Run this once in **Engineering Development** after 0B. It demonstrates the public FabricOps I/O helpers and prepares the managed source tables required by `02_pipeline`.


## 1. Load the shared Fabric configuration

The setup notebook uses the same workspace-local `00_env_config` as the rest of the Guided Demo.


In [ ]:
%run 00_env_config


## 2. Import the FabricOps I/O helpers


In [ ]:
from fabricops_kit import (
    read_lakehouse_csv,
    read_lakehouse_excel,
    read_lakehouse_json,
    read_lakehouse_parquet,
    read_lakehouse_table,
    read_warehouse_query,
    read_warehouse_table,
    write_lakehouse_table,
    write_warehouse_table,
)


## 3. Read the canonical Orders data from four file formats

All four files contain the same 120 logical Orders rows. Paths are relative to the configured Lakehouse `Files` area, so `Demo/orders.csv` resolves under `Files/Demo/orders.csv`.


In [ ]:
orders_csv_df = read_lakehouse_csv(
    "Demo/orders.csv",
    store="Bronze",
    spark_session=spark,
    header=True,
    inferSchema=True,
)

orders_json_df = read_lakehouse_json(
    "Demo/orders.json",
    store="Bronze",
    spark_session=spark,
)

orders_parquet_df = read_lakehouse_parquet(
    "Demo/orders.parquet",
    store="Bronze",
    spark_session=spark,
)

orders_excel_df = read_lakehouse_excel(
    "Demo/orders.xlsx",
    store="Bronze",
    spark_session=spark,
)


### Quick equivalence check

This is a simple walkthrough check, not a replacement for the repository tests.


In [ ]:
format_counts = {
    "csv": orders_csv_df.count(),
    "json": orders_json_df.count(),
    "parquet": orders_parquet_df.count(),
    "excel": orders_excel_df.count(),
}

print(format_counts)
assert len(set(format_counts.values())) == 1, "Orders file variants should have the same row count."
assert format_counts["csv"] == 120, "The canonical Day 1 Orders baseline should contain 120 rows."


## 4. Read the other demo inputs

`products.csv` becomes the Lakehouse lookup table. `order_history.csv` becomes the Warehouse history table.


In [ ]:
products_df = read_lakehouse_csv(
    "Demo/products.csv",
    store="Bronze",
    spark_session=spark,
    header=True,
    inferSchema=True,
)

order_history_df = read_lakehouse_csv(
    "Demo/order_history.csv",
    store="Bronze",
    spark_session=spark,
    header=True,
    inferSchema=True,
)

print(f"products rows: {products_df.count()}")
print(f"order_history rows: {order_history_df.count()}")


## 5. Write the managed tables used by `02_pipeline`

The CSV Orders read is the canonical Day 1 baseline used for the managed source table.


In [ ]:
write_lakehouse_table(
    orders_csv_df,
    table_name="orders",
    store="Bronze",
    schema="demo",
    mode="overwrite",
)

write_lakehouse_table(
    products_df,
    table_name="products",
    store="Bronze",
    schema="demo",
    mode="overwrite",
)

write_warehouse_table(
    order_history_df,
    schema="demo",
    table_name="order_history",
    store="Gold",
    mode="overwrite",
)


## 6. Read the Lakehouse tables back

These reads prove the managed Lakehouse sources are available through the same logical store configuration.


In [ ]:
orders_table_df = read_lakehouse_table(
    table_name="orders",
    store="Bronze",
    schema="demo",
    spark_session=spark,
)

products_table_df = read_lakehouse_table(
    table_name="products",
    store="Bronze",
    schema="demo",
    spark_session=spark,
)

print(f"managed orders rows: {orders_table_df.count()}")
print(f"managed products rows: {products_table_df.count()}")


## 7. Read the Warehouse back

`read_warehouse_table()` reads the full table. `read_warehouse_query()` lets the Warehouse execute the SQL first so filters and projections are pushed down before Spark receives the result.


In [ ]:
order_history_table_df = read_warehouse_table(
    schema="demo",
    table_name="order_history",
    store="Gold",
    spark_session=spark,
)

order_history_sample_df = read_warehouse_query(
    "SELECT TOP 5 * FROM demo.order_history ORDER BY 1",
    store="Gold",
    spark_session=spark,
)

print(f"warehouse order_history rows: {order_history_table_df.count()}")
display(order_history_sample_df)


## 8. Later demo assets — do not load yet

`Demo/orders_incremental.csv` is intentionally left untouched. It is revisited later in the `02_pipeline` walkthrough to simulate a later source arrival.

The partition and watermark fixtures are also left untouched for the later incremental/load-strategy showcase.

`Demo/orders_guardrail_failures.csv` is left untouched until the later Guardrail validation story. Keeping the initial baseline valid makes the first Engineering run deterministic.


## Ready for the Guided Demo

The setup is complete when these managed tables exist:

- `bronze.demo.orders`
- `bronze.demo.products`
- `gold.demo.order_history`

Continue with Step 1 in `01_governance`, then Step 2 in `02_pipeline`.
